# 01 — Data Profiling: Online Retail II

Notebook ini melakukan **profiling awal** terhadap dataset `Online Retail II` sebelum proses cleaning.

### Tujuan
- Memastikan dataset memenuhi skala tugas Big Data.
- Memahami struktur kolom, tipe data, dan rentang waktu.
- Mengukur missing values, duplicate rows, cancellation/return, serta nilai numerik tidak valid.
- Menyimpan cache data gabungan agar notebook berikutnya lebih cepat.

> **Urutan pengerjaan:** jalankan notebook ini terlebih dahulu, kemudian `02_data_cleaning.ipynb`, lalu `03_eda_and_insights.ipynb`.

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Menentukan lokasi project dan dataset

Kode di bawah dibuat fleksibel supaya tetap bekerja ketika notebook dibuka dari:

- `/home/jovyan/work` (Docker/Jupyter), atau
- direktori project secara langsung.

Dataset dicari dengan pola `online_retail_II*.xlsx`.

In [4]:
from pathlib import Path

def find_project_root():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/home/jovyan/work"),
    ]

    # Hilangkan path duplikat
    candidates = list(dict.fromkeys(candidates))

    for base in candidates:
        if not base.exists():
            continue

        # Lokasi yang sesuai struktur project kamu
        raw_dir = base / "data" / "raw"

        if raw_dir.exists():
            files = [
                p for p in raw_dir.glob("online_retail_II*.xlsx")
                if not p.name.startswith("~$")
            ]

            if files:
                return base.resolve(), sorted(files)[0].resolve()

        # Fallback kalau dataset ternyata ada langsung di root
        files = [
            p for p in base.glob("online_retail_II*.xlsx")
            if not p.name.startswith("~$")
        ]

        if files:
            return base.resolve(), sorted(files)[0].resolve()

    raise FileNotFoundError(
        "Dataset online_retail_II*.xlsx tidak ditemukan di data/raw."
    )


ROOT, RAW_FILE = find_project_root()

INTERIM_DIR = ROOT / "data" / "interim"
OUTPUT_DIR = ROOT / "outputs" / "profiling"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Dataset      :", RAW_FILE)
print("Ukuran file :", f"{RAW_FILE.stat().st_size / 1024**2:,.2f} MiB")

Project root : /home/jovyan/work
Dataset      : /home/jovyan/work/data/raw/online_retail_II.xlsx
Ukuran file : 43.51 MiB


## 2. Memeriksa sheet Excel

In [5]:
xls = pd.ExcelFile(RAW_FILE, engine="openpyxl")
print("Daftar sheet:")
for sheet in xls.sheet_names:
    print("-", sheet)

Daftar sheet:
- Year 2009-2010
- Year 2010-2011


## 3. Membaca dan menggabungkan seluruh sheet

Kolom `SourceYear` ditambahkan agar asal sheet setiap baris tetap dapat dilacak.

In [6]:
frames = []

for sheet in xls.sheet_names:
    print(f"Membaca sheet: {sheet} ...")
    temp = pd.read_excel(RAW_FILE, sheet_name=sheet, engine="openpyxl")
    temp["SourceYear"] = sheet
    frames.append(temp)

df = pd.concat(frames, ignore_index=True)

print("\nSelesai.")
print(f"Jumlah baris : {len(df):,}")
print(f"Jumlah kolom : {df.shape[1]:,}")
display(df.head())

Membaca sheet: Year 2009-2010 ...
Membaca sheet: Year 2010-2011 ...

Selesai.
Jumlah baris : 1,067,371
Jumlah kolom : 9


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010


## 4. Ringkasan ukuran dataset

Dataset asli mempunyai 8 kolom transaksi. Setelah digabungkan, notebook menambahkan satu kolom `SourceYear`.

Karena file Excel memiliki dua sheet, total jumlah record dihitung setelah kedua sheet digabung.

In [7]:
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

dataset_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Memory usage in pandas (MiB)",
        "Excel file size (MiB)"
    ],
    "Value": [
        f"{len(df):,}",
        f"{df.shape[1]:,}",
        f"{memory_mb:,.2f}",
        f"{RAW_FILE.stat().st_size / 1024**2:,.2f}"
    ]
})

display(dataset_summary)

,Metric,Value
0,Rows,"1,067,371"
1,Columns,9
2,Memory usage in pandas (MiB),229.84
3,Excel file size (MiB),43.51


## 5. Struktur kolom dan tipe data

In [8]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "non_null": [df[col].notna().sum() for col in df.columns],
    "null": [df[col].isna().sum() for col in df.columns],
    "unique": [df[col].nunique(dropna=True) for col in df.columns],
})

display(schema)

,column,dtype,non_null,null,unique
0,Invoice,object,1067371,0,53628
1,StockCode,object,1067371,0,5305
2,Description,object,1062989,4382,5698
3,Quantity,int64,1067371,0,1057
4,InvoiceDate,datetime64[us],1067371,0,47635
5,Price,float64,1067371,0,2807
6,Customer ID,float64,824364,243007,5942
7,Country,str,1067371,0,43
8,SourceYear,str,1067371,0,2


## 6. Missing values

Persentase missing value membantu menentukan aturan cleaning pada notebook berikutnya.

In [9]:
missing = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
)

missing["missing_pct"] = missing["missing_count"] / len(df) * 100
missing = missing.sort_values("missing_count", ascending=False)

display(missing)

,missing_count,missing_pct
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
Quantity,0,0.00
StockCode,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
SourceYear,0,0.00


## 7. Duplicate rows

In [10]:
duplicate_count = df.duplicated().sum()
duplicate_pct = duplicate_count / len(df) * 100

print(f"Exact duplicate rows : {duplicate_count:,}")
print(f"Duplicate percentage : {duplicate_pct:.4f}%")

Exact duplicate rows : 12,133
Duplicate percentage : 1.1367%


## 8. Statistik numerik

In [11]:
numeric_cols = [c for c in ["Quantity", "Price", "Customer ID"] if c in df.columns]
display(df[numeric_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
Quantity,"1,067,371.00",9.94,172.71,"-80,995.00",1.00,3.00,10.00,"80,995.00"
Price,"1,067,371.00",4.65,123.55,"-53,594.36",1.25,2.10,4.15,"38,970.00"
Customer ID,"824,364.00","15,324.64","1,697.46","12,346.00","13,975.00","15,255.00","16,797.00","18,287.00"


## 9. Pemeriksaan kualitas transaksi

Pada Online Retail II:
- invoice yang diawali `C` umumnya menunjukkan cancellation,
- `Quantity <= 0` biasanya berkaitan dengan return/cancellation,
- `Price <= 0` perlu diperiksa sebelum analisis penjualan.

In [12]:
invoice_str = df["Invoice"].astype("string")

quality_checks = pd.Series({
    "cancelled_invoice_prefix_C": invoice_str.str.startswith("C", na=False).sum(),
    "quantity_le_0": (pd.to_numeric(df["Quantity"], errors="coerce") <= 0).sum(),
    "price_le_0": (pd.to_numeric(df["Price"], errors="coerce") <= 0).sum(),
    "missing_description": df["Description"].isna().sum(),
    "missing_customer_id": df["Customer ID"].isna().sum(),
    "exact_duplicates": df.duplicated().sum(),
}, name="count").to_frame()

quality_checks["pct_of_rows"] = quality_checks["count"] / len(df) * 100
display(quality_checks)

,count,pct_of_rows
cancelled_invoice_prefix_C,19494,1.83
quantity_le_0,22950,2.15
price_le_0,6207,0.58
missing_description,4382,0.41
missing_customer_id,243007,22.77
exact_duplicates,12133,1.14


## 10. Rentang waktu dan distribusi per tahun

In [13]:
invoice_date = pd.to_datetime(df["InvoiceDate"], errors="coerce")

print("Tanggal minimum :", invoice_date.min())
print("Tanggal maksimum:", invoice_date.max())

year_counts = invoice_date.dt.year.value_counts().sort_index().rename("rows").to_frame()
display(year_counts)

Tanggal minimum : 2009-12-01 07:45:00
Tanggal maksimum: 2011-12-09 12:50:00


,rows
InvoiceDate,
2009,45228
2010,522714
2011,499429


## 11. Ringkasan negara

In [14]:
country_profile = (
    df["Country"]
      .fillna("<MISSING>")
      .value_counts()
      .rename_axis("Country")
      .reset_index(name="Rows")
)

print("Jumlah negara unik:", df["Country"].nunique(dropna=True))
display(country_profile.head(15))

Jumlah negara unik: 43


,Country,Rows
0,United Kingdom,981330
1,EIRE,17866
2,Germany,17624
3,France,14330
4,Netherlands,5140
5,Spain,3811
6,Switzerland,3189
7,Belgium,3123
8,Portugal,2620
9,Australia,1913


## 12. Produk/deskripsi yang paling sering muncul

Bagian ini masih **profiling frekuensi baris**, bukan ranking penjualan. Ranking revenue dilakukan setelah cleaning.

In [15]:
top_descriptions = (
    df["Description"]
      .fillna("<MISSING>")
      .value_counts()
      .head(15)
      .rename_axis("Description")
      .reset_index(name="Rows")
)

display(top_descriptions)

,Description,Rows
0,WHITE HANGING HEART T-LIGHT HOLDER,5918
1,REGENCY CAKESTAND 3 TIER,4412
2,<MISSING>,4382
3,JUMBO BAG RED RETROSPOT,3469
4,ASSORTED COLOUR BIRD ORNAMENT,2958
5,PARTY BUNTING,2765
6,STRAWBERRY CERAMIC TRINKET BOX,2613
7,LUNCH BAG BLACK SKULL.,2529
8,JUMBO STORAGE BAG SUKI,2434
9,HEART OF WICKER SMALL,2319


## 13. Simpan hasil profiling dan cache data mentah gabungan

Cache Parquet diprioritaskan karena lebih cepat dan lebih hemat ruang dibanding membaca ulang Excel.

Jika `pyarrow` tidak tersedia, notebook otomatis menyimpan CSV.

In [17]:
# Simpan output profiling
schema.to_csv(OUTPUT_DIR / "schema.csv", index=False)

missing.reset_index(names="column").to_csv(
    OUTPUT_DIR / "missing_values.csv",
    index=False
)

quality_checks.reset_index(names="check").to_csv(
    OUTPUT_DIR / "quality_checks.csv",
    index=False
)

country_profile.to_csv(
    OUTPUT_DIR / "country_profile.csv",
    index=False
)


df_save = df.copy()

# Identifier/text harus konsisten sebagai string.
# Invoice mengandung kode seperti C489449, jadi tidak boleh integer.
string_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Country",
    "SourceYear",
]

for col in string_columns:
    if col in df_save.columns:
        df_save[col] = df_save[col].astype("string")

# Numeric
df_save["Quantity"] = pd.to_numeric(
    df_save["Quantity"],
    errors="coerce"
).astype("Int64")

df_save["Price"] = pd.to_numeric(
    df_save["Price"],
    errors="coerce"
)

df_save["Customer ID"] = pd.to_numeric(
    df_save["Customer ID"],
    errors="coerce"
).astype("Int64")

# Datetime
df_save["InvoiceDate"] = pd.to_datetime(
    df_save["InvoiceDate"],
    errors="coerce"
)


raw_parquet = INTERIM_DIR / "online_retail_raw.parquet"

df_save.to_parquet(
    raw_parquet,
    index=False
)

print("✅ Cache Parquet berhasil disimpan:")
print(raw_parquet)

print(
    "Ukuran:",
    f"{raw_parquet.stat().st_size / 1024**2:,.2f} MiB"
)

✅ Cache Parquet berhasil disimpan:
/home/jovyan/work/data/interim/online_retail_raw.parquet
Ukuran: 6.90 MiB


## 14. Kesimpulan profiling

Gunakan output di atas untuk menjelaskan kondisi awal dataset. Poin yang perlu dicatat dalam laporan:

1. Total record setelah dua sheet digabung.
2. Kolom yang memiliki missing value terbesar.
3. Jumlah exact duplicate.
4. Jumlah cancellation/return.
5. Keberadaan `Quantity <= 0` dan `Price <= 0`.
6. Rentang waktu transaksi dan negara yang mendominasi.

Cleaning pada notebook berikutnya **tidak langsung membuang semua missing Customer ID**, karena transaksi tanpa Customer ID masih dapat berguna untuk analisis penjualan agregat. Untuk analisis customer, baris tanpa Customer ID akan difilter saat dibutuhkan.